# 0. Read files and Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings
import os

warnings.filterwarnings('ignore')

# Device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load data
print("Loading data...")

input_path = 'trademaster25' if os.path.exists('trademaster25') else '/kaggle/input/trademaster'

train_df = pd.read_csv(f'{input_path}/train_v2.csv')
test_df = pd.read_csv(f'{input_path}/test_v2.csv')

print(f"Training set: {train_df.shape}")
print(f"Test set: {test_df.shape}")

# Feature and target columns
feature_cols = [f'feature_{i}' for i in range(1, 27)]+[f'feature_{j}' for j in range(28, 31)]
target_cols = ['target_short', 'target_medium', 'target_long']
time_cols = ['date_id', 'minute_id']
TARGET_WEIGHTS = {'short': 0.5, 'medium': 0.3, 'long': 0.2}

print(f"Features: {len(feature_cols)}")
print(f"Targets: {target_cols}")
print(f"Weights: {TARGET_WEIGHTS}")

In [ ]:
# Fill NaN with median and clip extreme values
for col in feature_cols:
    median_val = train_df[col].median()
    train_df[col].fillna(median_val, inplace=True)
    test_df[col].fillna(median_val, inplace=True)

    # Clip extreme values at 1% and 99% quantiles
    lower = train_df[col].quantile(0.01)
    upper = train_df[col].quantile(0.99)
    train_df[col] = train_df[col].clip(lower, upper)
    test_df[col] = test_df[col].clip(lower, upper)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.preprocessing import QuantileTransformer

print("=" * 80)
print("识别连续特征和离散特征")
print("=" * 80)

# 识别离散特征（唯一值较少的特征）
discrete_features = []
continuous_features = []

for col in feature_cols:
    unique_count = train_df[col].nunique()
    unique_vals = train_df[col].unique()
    
    # 如果唯一值 <= 10 且都是整数，认为是离散特征
    if unique_count <= 10:
        is_integer = all(train_df[col].dropna().apply(lambda x: x == int(x)))
        if is_integer or unique_count <= 3:
            discrete_features.append(col)
            print(f"{col}: 离散特征 (唯一值={unique_count}, 值={sorted(unique_vals[:10])})")
        else:
            continuous_features.append(col)
    else:
        continuous_features.append(col)

print(f"\n发现 {len(discrete_features)} 个离散特征")
print(f"发现 {len(continuous_features)} 个连续特征")
print(f"\n离散特征列表: {discrete_features}")

print("\n" + "=" * 80)
print("GaussRank 归一化 (仅对连续特征)")
print("=" * 80)

if len(continuous_features) > 0:
    # 对连续特征进行 GaussRank 归一化
    gauss_scaler = QuantileTransformer(output_distribution='normal', random_state=42)
    
    print(f"对 {len(continuous_features)} 个连续特征应用 GaussRank 归一化...")
    train_df[continuous_features] = gauss_scaler.fit_transform(train_df[continuous_features])
    test_df[continuous_features] = gauss_scaler.transform(test_df[continuous_features])
    
    print("GaussRank 归一化完成。")
    
    # 验证归一化后的分布
    print("\n归一化后的偏度和峰度 (前5个连续特征):")
    print(train_df[continuous_features[:5]].agg(['skew', 'kurtosis']))

if len(discrete_features) > 0:
    print("\n" + "=" * 80)
    print("离散特征处理")
    print("=" * 80)
    print(f"离散特征保持原样，不进行归一化")
    print(f"这些特征的离散性质在树模型中会被保留")
    
    # 显示离散特征的分布
    print("\n离散特征分布:")
    for col in discrete_features[:5]:  # 只显示前5个
        value_counts = train_df[col].value_counts().sort_index()
        print(f"\n{col}:")
        print(value_counts)


### 1. feature engineering

In [ ]:
# NOTE: Run this cell before training XGBoost-based models.
from typing import List

print("Applying feature engineering for XGBoost (stationarity, time features, interactions)...")

# Helper configuration
feature_6_min = min(train_df['feature_6'].min(), test_df['feature_6'].min()) if 'feature_6' in train_df.columns else 0
feature_6_max = max(train_df['feature_6'].max(), test_df['feature_6'].max()) if 'feature_6' in train_df.columns else 0

# Columns to use when computing group-wise differences (if available)
group_candidates: List[str] = [
    col for col in ['date_id', 'time_id', 'stock_id'] if col in train_df.columns
]

# Ensure feature column list exists
if 'feature_cols' not in globals():
    feature_cols = [c for c in train_df.columns if c.startswith('feature_')]

# 1. Stationary transformation for feature_11 -> f11_diff (drop original)
for df_name, df in [('train', train_df), ('test', test_df)]:
    if 'feature_11' in df.columns:
        if group_candidates:
            df['f11_diff'] = df.groupby(group_candidates)['feature_11'].diff()
        else:
            df['f11_diff'] = df['feature_11'].diff()
        df['f11_diff'].fillna(0.0, inplace=True)
        df.drop(columns='feature_11', inplace=True)
    elif 'f11_diff' not in df.columns:
        df['f11_diff'] = 0.0

# 2. Drop low-value raw feature_27
for df in [train_df, test_df]:
    if 'feature_27' in df.columns:
        df.drop(columns='feature_27', inplace=True)

# 3. Intraday time derivatives from feature_6
if 'feature_6' in train_df.columns:
    for df_name, df in [('train', train_df), ('test', test_df)]:
        df['dist_to_open'] = df['feature_6'] - feature_6_min
        df['dist_to_close'] = feature_6_max - df['feature_6']
        df['is_midday'] = ((df['feature_6'] > 85) & (df['feature_6'] < 190)).astype(int)
        df['volatility_time_norm'] = df['feature_4'] / (df['feature_6'] - feature_6_min + 10)
        df['late_session_flow'] = df['feature_4'] * (df['feature_6'] > 200).astype(int)

# 4. Log transforms to spread long-tailed signals
for col in ['feature_4', 'feature_13', 'feature_14']:
    if col in train_df.columns:
        train_df[f'{col}_log1p'] = np.log1p(np.clip(train_df[col], a_min=0, a_max=None))
        test_df[f'{col}_log1p'] = np.log1p(np.clip(test_df[col], a_min=0, a_max=None))

# 5. Ratio-style features for risk/liquidity interpretation
if 'feature_4' in train_df.columns and 'f11_diff' in train_df.columns:
    denom = lambda x: x.replace(0, np.nan)
    train_df['f11_diff_div_f4'] = train_df['f11_diff'] / denom(train_df['feature_4'])
    test_df['f11_diff_div_f4'] = test_df['f11_diff'] / denom(test_df['feature_4'])
    train_df['f11_diff_div_f4'].replace([np.inf, -np.inf], 0.0, inplace=True)
    test_df['f11_diff_div_f4'].replace([np.inf, -np.inf], 0.0, inplace=True)
    train_df['f11_diff_div_f4'].fillna(0.0, inplace=True)
    test_df['f11_diff_div_f4'].fillna(0.0, inplace=True)

if 'feature_13' in train_df.columns and 'feature_4' in train_df.columns:
    denom = lambda x: x.replace(0, np.nan)
    train_df['feature_13_div_f4'] = train_df['feature_13'] / denom(train_df['feature_4'])
    test_df['feature_13_div_f4'] = test_df['feature_13'] / denom(test_df['feature_4'])
    for df in [train_df, test_df]:
        df['feature_13_div_f4'].replace([np.inf, -np.inf], 0.0, inplace=True)
        df['feature_13_div_f4'].fillna(0.0, inplace=True)

# 6. Categorical handling for low-cardinality features
for cat_col in ['feature_1', 'feature_8', 'feature_12']:
    if cat_col in train_df.columns:
        train_df[cat_col] = train_df[cat_col].astype('category')
        test_df[cat_col] = test_df[cat_col].astype('category')

# Count encoding for feature_8 (using train distribution)
if 'feature_8' in train_df.columns:
    feature_8_counts = train_df['feature_8'].value_counts()
    for df in [train_df, test_df]:
        mapped_counts = df['feature_8'].map(feature_8_counts).astype('float64')
        df.drop(columns=['feature_8_count'], errors='ignore', inplace=True)
        df['feature_8_count'] = mapped_counts.fillna(0.0)

# 7. Update feature column registry
new_features = [
    'f11_diff', 'dist_to_open', 'dist_to_close', 'is_midday',
    'volatility_time_norm', 'late_session_flow',
    'feature_4_log1p', 'feature_13_log1p', 'feature_14_log1p',
    'f11_diff_div_f4', 'feature_13_div_f4', 'feature_8_count'
]

dropped_features = ['feature_11', 'feature_27']
feature_cols = [col for col in feature_cols if col not in dropped_features]
for extra in new_features:
    if extra not in feature_cols:
        feature_cols.append(extra)

print(f"Updated feature set size: {len(feature_cols)}")

# XGBoost+MLP

In [ ]:
split_idx = int(len(train_df) * 0.8)
print(f"Train/Val Split Index: {split_idx}")

X_train = train_df[feature_cols].iloc[:split_idx].values
y_train = train_df[target_cols].iloc[:split_idx].values
X_val = train_df[feature_cols].iloc[split_idx+960:].values
y_val = train_df[target_cols].iloc[split_idx+960:].values
X_test = test_df[feature_cols].values

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
print("="*80)
print("Training XGBoost Models")
print("="*80)

xgb_models = {}
xgb_val_predictions = {}

for i, target_name in enumerate(['short', 'medium', 'long']):
    print(f"\n--- Training XGBoost for target_{target_name} ---")
    
    # XGBoost parameters with early stopping
    xgb_params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'max_depth': 8,
        'learning_rate': 0.05,
        'n_estimators': 1000,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0,
        'reg_lambda': 1e-5,
        'random_state': 42,
        'eval_metric': 'mae',
        'early_stopping_rounds': 50
    }
    
    model = xgb.XGBRegressor(**xgb_params)
    
    # Train
    model.fit(
        X_train, y_train[:, i],
        eval_set=[(X_val, y_val[:, i])],
        verbose=100
    )
    
    # Predict on validation set
    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val[:, i], val_pred)
    
    print(f"Best iteration: {model.best_iteration}")
    print(f"Validation MAE: {val_mae:.6f}")
    
    # Store model and predictions
    xgb_models[target_name] = model
    xgb_val_predictions[target_name] = val_pred

# Calculate weighted MAE for XGBoost
xgb_weighted_mae = (
    TARGET_WEIGHTS['short'] * mean_absolute_error(y_val[:, 0], xgb_val_predictions['short']) +
    TARGET_WEIGHTS['medium'] * mean_absolute_error(y_val[:, 1], xgb_val_predictions['medium']) +
    TARGET_WEIGHTS['long'] * mean_absolute_error(y_val[:, 2], xgb_val_predictions['long'])
)

print("\n" + "="*80)
print("XGBoost Summary")
print("="*80)
print(f"Short MAE:  {mean_absolute_error(y_val[:, 0], xgb_val_predictions['short']):.6f} (weight: 0.5)")
print(f"Medium MAE: {mean_absolute_error(y_val[:, 1], xgb_val_predictions['medium']):.6f} (weight: 0.3)")
print(f"Long MAE:   {mean_absolute_error(y_val[:, 2], xgb_val_predictions['long']):.6f} (weight: 0.2)")
print(f"Weighted MAE: {xgb_weighted_mae:.6f}")
print("="*80)

# 专门预测 target_long 的 XGBoost 模型

In [ ]:
print("="*80)
print("训练专门预测 target_long 的 XGBoost 模型")
print("="*80)

# 准备 target_long 的数据
y_train_long = y_train[:, 2]  # target_long 是第3个目标（索引2）
y_val_long = y_val[:, 2]

# XGBoost 参数优化（专门针对 long 目标）
xgb_long_params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'max_depth': 8,  # 增加深度以捕捉更复杂的模式
    'learning_rate': 0.005,  # 降低学习率以更精细地学习
    'n_estimators': 2000,  # 增加迭代次数
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'colsample_bylevel': 0.9,
    'colsample_bynode': 0.9,
    'reg_alpha': 0.05,  # L1 正则化
    'reg_lambda': 2.0,  # L2 正则化
    'min_child_weight': 3,
    'gamma': 0.1,
    'random_state': 42,
    'eval_metric': 'mae',
    'early_stopping_rounds': 100  # 增加早停轮次
}

# 创建并训练模型
print("\n开始训练 XGBoost for target_long...")
model_long = xgb.XGBRegressor(**xgb_long_params)

# 训练模型
model_long.fit(
    X_train, y_train_long,
    eval_set=[(X_val, y_val_long)],
    verbose=100
)

# 在验证集上进行预测
val_pred_long = model_long.predict(X_val)
mae_long = mean_absolute_error(y_val_long, val_pred_long)

# 在测试集上进行预测
test_pred_xgb_long = model_long.predict(X_test)

print("\n" + "="*80)
print("专门的 target_long XGBoost 模型结果")
print("="*80)
print(f"最佳迭代轮次: {model_long.best_iteration}")
print(f"验证集 MAE: {mae_long:.6f}")
print(f"验证集预测统计:")
print(f"  - 均值: {val_pred_long.mean():.6f}")
print(f"  - 标准差: {val_pred_long.std():.6f}")
print(f"  - 最小值: {val_pred_long.min():.6f}")
print(f"  - 最大值: {val_pred_long.max():.6f}")
print(f"\n验证集真实值统计:")
print(f"  - 均值: {y_val_long.mean():.6f}")
print(f"  - 标准差: {y_val_long.std():.6f}")
print(f"  - 最小值: {y_val_long.min():.6f}")
print(f"  - 最大值: {y_val_long.max():.6f}")
print(f"\n测试集预测统计:")
print(f"  - 均值: {test_pred_xgb_long.mean():.6f}")
print(f"  - 标准差: {test_pred_xgb_long.std():.6f}")
print(f"  - 最小值: {test_pred_xgb_long.min():.6f}")
print(f"  - 最大值: {test_pred_xgb_long.max():.6f}")
print("="*80)

# 特征重要性分析
print("\n前20个最重要的特征 (for target_long):")
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_long.feature_importances_
}).sort_values('importance', ascending=False)
print(feature_importance.head(20))

In [ ]:
print("=" * 80)
print("使用 Optuna 调优 XGBoost 参数（优化 weighted MAE）")
print("=" * 80)

import optuna
import xgboost as xgb
import torch
from sklearn.metrics import mean_absolute_error

TARGET_ORDER = ["short", "medium", "long"]

def objective(trial):
    lr = trial.suggest_float("learning_rate", 0.01, 0.1, log=True)
    n_estimators = int(300 / lr)  # 👈 强制耦合
    
    params = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "device": "cuda" if torch.cuda.is_available() else "cpu",

        # Optuna 搜索空间
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": lr,
        "n_estimators": min(n_estimators, 2500),
        "subsample": trial.suggest_float("subsample", 0.5, 0.85),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 5, 50, log=True),

        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 20.0, log=True),

        "eval_metric": "mae",
        "random_state": 42,
    }

    maes = []

    for idx, name in enumerate(TARGET_ORDER):
        model = xgb.XGBRegressor(**params)

        model.fit(
            X_train,
            y_train[:, idx],
            eval_set=[(X_val, y_val[:, idx])],
            verbose=False
        )

        val_pred = model.predict(X_val)
        maes.append(mean_absolute_error(y_val[:, idx], val_pred))

    weighted_mae = (
        TARGET_WEIGHTS["short"] * maes[0]
        + TARGET_WEIGHTS["medium"] * maes[1]
        + TARGET_WEIGHTS["long"] * maes[2]
    )

    return weighted_mae


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

best_params = study.best_params
print(f"\n最佳 weighted MAE: {study.best_value:.6f}")
print("最佳参数:")
for k, v in best_params.items():
    print(f"  {k}: {v}")


print("\n" + "=" * 80)
print("使用最佳参数训练 XGBoost 模型")
print("=" * 80)

xgb_models = {}
xgb_val_predictions = {}

fixed_params = {
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "eval_metric": "mae",
    "random_state": 42,
}

final_params = {**fixed_params, **best_params}

for idx, target_name in enumerate(TARGET_ORDER):
    print(f"\n--- 训练 target_{target_name} 的 XGBoost ---")

    model = xgb.XGBRegressor(**final_params)
    model.fit(
        X_train,
        y_train[:, idx],
        eval_set=[(X_val, y_val[:, idx])],
        verbose=100
    )

    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val[:, idx], val_pred)

    print(f"验证集 MAE: {val_mae:.6f}")

    xgb_models[target_name] = model
    xgb_val_predictions[target_name] = val_pred


xgb_weighted_mae = (
    TARGET_WEIGHTS["short"] * mean_absolute_error(y_val[:, 0], xgb_val_predictions["short"])
    + TARGET_WEIGHTS["medium"] * mean_absolute_error(y_val[:, 1], xgb_val_predictions["medium"])
    + TARGET_WEIGHTS["long"] * mean_absolute_error(y_val[:, 2], xgb_val_predictions["long"])
)

print("\n" + "="*80)
print("XGBoost 调优后总结")
print("="*80)
print(
    f"Short MAE:  {mean_absolute_error(y_val[:, 0], xgb_val_predictions['short']):.6f} (权重: {TARGET_WEIGHTS['short']})"
)
print(
    f"Medium MAE: {mean_absolute_error(y_val[:, 1], xgb_val_predictions['medium']):.6f} (权重: {TARGET_WEIGHTS['medium']})"
)
print(
    f"Long MAE:   {mean_absolute_error(y_val[:, 2], xgb_val_predictions['long']):.6f} (权重: {TARGET_WEIGHTS['long']})"
)
print(f"Weighted MAE: {xgb_weighted_mae:.6f}")
print("="*80)


In [ ]:
print(pd.Series(val_pred).describe())

In [ ]:
# Standardize features for neural network (don't scale targets)
print("Standardizing features...")
from sklearn.preprocessing import RobustScaler

scaler_X = RobustScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

print("Standardization complete")

class MultiTargetDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y) if y is not None else None
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# Create datasets (use original y_train, y_val - no scaling)
train_dataset = MultiTargetDataset(X_train_scaled, y_train)
val_dataset = MultiTargetDataset(X_val_scaled, y_val)
test_dataset = MultiTargetDataset(X_test_scaled)

# Create dataloaders
BATCH_SIZE = 512*8*8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

class MLPMultiTarget(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64], num_targets=3, dropout=0.2):
        super(MLPMultiTarget, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        self.shared_layers = nn.Sequential(*layers)
        
        # Output layer: predict all 3 targets simultaneously
        self.output_layer = nn.Linear(prev_dim, num_targets)
    
    def forward(self, x):
        x = self.shared_layers(x)
        return self.output_layer(x)

# Initialize model
INPUT_DIM = X_train_scaled.shape[1]
mlp_model = MLPMultiTarget(input_dim=INPUT_DIM).to(DEVICE)

print(f"MLP Model:")
print(mlp_model)
print(f"\nTotal parameters: {sum(p.numel() for p in mlp_model.parameters()):,}")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 2:
                X_batch, y_batch = batch
                X_batch = X_batch.to(device)
                
                outputs = model(X_batch)
                
                all_preds.append(outputs.cpu().numpy())
                all_labels.append(y_batch.numpy())
            else:
                X_batch = batch.to(device)
                outputs = model(X_batch)
                all_preds.append(outputs.cpu().numpy())
    
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels) if len(all_labels) > 0 else None
    
    return all_preds, all_labels

print("Training functions defined")

In [ ]:
print("Training MLP model...")
print("="*80)

criterion = nn.L1Loss()  # Use MAE loss directly
optimizer = optim.AdamW(mlp_model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

EPOCHS = 20
best_weighted_mae = float('inf')
patience = 5
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss = train_epoch(mlp_model, train_loader, criterion, optimizer, DEVICE)
    val_preds, val_labels = eval_epoch(mlp_model, val_loader, DEVICE)
    
    # Calculate MAE for each target
    mae_short = mean_absolute_error(val_labels[:, 0], val_preds[:, 0])
    mae_medium = mean_absolute_error(val_labels[:, 1], val_preds[:, 1])
    mae_long = mean_absolute_error(val_labels[:, 2], val_preds[:, 2])
    
    # Calculate weighted MAE
    weighted_mae = (
        TARGET_WEIGHTS['short'] * mae_short +
        TARGET_WEIGHTS['medium'] * mae_medium +
        TARGET_WEIGHTS['long'] * mae_long
    )
    
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(weighted_mae)
    new_lr = optimizer.param_groups[0]['lr']
    
    if new_lr != old_lr:
        print(f"  Learning rate reduced: {old_lr:.6f} -> {new_lr:.6f}")
    
    if weighted_mae < best_weighted_mae:
        best_weighted_mae = weighted_mae
        patience_counter = 0
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}:")
        print(f"  Train Loss: {train_loss:.6f}")
        print(f"  Val Weighted MAE: {weighted_mae:.6f} (Best: {best_weighted_mae:.6f})")
        print(f"    Short: {mae_short:.6f}, Medium: {mae_medium:.6f}, Long: {mae_long:.6f}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nMLP training complete")
print(f"Best validation Weighted MAE: {best_weighted_mae:.6f}")

In [ ]:
val_preds, val_labels = eval_epoch(mlp_model, val_loader, DEVICE)

mlp_weighted_mae = (
    TARGET_WEIGHTS['short'] * mean_absolute_error(val_labels[:, 0], val_preds[:, 0]) +
    TARGET_WEIGHTS['medium'] * mean_absolute_error(val_labels[:, 1], val_preds[:, 1]) +
    TARGET_WEIGHTS['long'] * mean_absolute_error(val_labels[:, 2], val_preds[:, 2])
)

print("="*80)
print("MLP Summary")
print("="*80)
print(f"Short MAE:  {mean_absolute_error(val_labels[:, 0], val_preds[:, 0]):.6f} (weight: 0.5)")
print(f"Medium MAE: {mean_absolute_error(val_labels[:, 1], val_preds[:, 1]):.6f} (weight: 0.3)")
print(f"Long MAE:   {mean_absolute_error(val_labels[:, 2], val_preds[:, 2]):.6f} (weight: 0.2)")
print(f"Weighted MAE: {mlp_weighted_mae:.6f}")
print("="*80)

In [ ]:
# Compare models
comparison = pd.DataFrame({
    'Model': ['XGBoost (3 models)', 'MLP (multi-target)'],
    'Weighted MAE': [xgb_weighted_mae, mlp_weighted_mae]
})

print(comparison.to_string(index=False))

best_model = 'XGBoost' if xgb_weighted_mae < mlp_weighted_mae else 'MLP'
print(f"\nBest model: {best_model}")

In [ ]:
print("Generating XGBoost predictions...")
test_pred_xgb_short = xgb_models['short'].predict(X_test)
test_pred_xgb_medium = xgb_models['medium'].predict(X_test)
test_pred_xgb_long = model_long.predict(X_test)

submission_xgb = pd.DataFrame({
    'id': test_df['id'],
    'target_short': test_pred_xgb_short,
    'target_medium': test_pred_xgb_medium,
    'target_long': test_pred_xgb_long
})
submission_xgb.to_csv('submission_xgb.csv', index=False)
print(f"✅ XGBoost submission saved: submission_xgb.csv")

# print("\nGenerating MLP predictions...")
# test_preds_mlp, _ = eval_epoch(mlp_model, test_loader, DEVICE)
# submission_mlp = pd.DataFrame({
#     'id': test_df['id'],
#     'target_short': test_preds_mlp[:, 0],
#     'target_medium': test_preds_mlp[:, 1],
#     'target_long': test_preds_mlp[:, 2]
# })
# submission_mlp.to_csv('submission.csv', index=False)
# print(f"✅ MLP submission saved: submission_mlp.csv")